# Terminology and Mathematical Language: Worked Solutions


## Problem 1. Evaluate a fixed probability model

**Problem.** For the four input rows below, evaluate the affine scores $s=Xw+b$ and probabilities $q_i=1/(1+e^{-s_i})$. Identify which quantities are supplied and which are computed.

**Solution.** The supplied parameters are $w=(0.7,-0.2)$ and $b=0.1$; the observations and binary targets are also given. Matrix multiplication gives scores $(-0.1,0.6,1.7,-0.6)$. Applying the sigmoid coordinatewise gives one probability per row. This calculation uses a fixed model; it estimates no parameters.


In [ ]:
import numpy as np
X = np.array([[0., 1.], [1., 1.], [2., -1.], [-1., 0.]])
w = np.array([0.7, -0.2])
b = 0.1
y = np.array([0., 1., 1., 0.])


In [ ]:
def probabilities(X, w, b):
    score = np.asarray(X) @ np.asarray(w) + b
    return 1. / (1. + np.exp(-score))


In [ ]:
q = probabilities(X, w, b)
assert q.shape == (4,)
assert np.all((q > 0) & (q < 1))
np.testing.assert_allclose(probabilities(np.zeros((3, 2)), w, 0.), 0.5)
np.testing.assert_allclose(q, 1/(1+np.exp(-np.array([-0.1, 0.6, 1.7, -0.6]))))


## Problem 2. Compute an empirical squared risk

**Problem.** Compute the mean squared loss between the given targets and probability predictions. Reject shapes that would turn paired differences into a pairwise table.

**Solution.** With target-first notation, $\ell(y_i,q_i)=(y_i-q_i)^2$, so $\widehat R=\frac14\sum_{i=1}^4(y_i-q_i)^2$. The target and prediction must both have the same nonempty vector shape. Subtracting shapes `(4,)` and `(4,1)` would instead broadcast to `(4,4)` and compare each target with every prediction. The checks reject that different calculation. The reported value is the loss on these four observations; the unknown population law is not supplied, so this value is not a population-risk guarantee.


In [ ]:
def empirical_squared_risk(target, prediction):
    target, prediction = np.asarray(target), np.asarray(prediction)
    if target.ndim != 1 or prediction.shape != target.shape or target.size == 0:
        raise ValueError("expected equal nonempty vector shapes")
    return float(np.mean((target - prediction)**2))


In [ ]:
assert empirical_squared_risk(np.array([0., 1.]), np.array([0.5, 0.5])) == 0.25
assert empirical_squared_risk(y, y) == 0.
try:
    empirical_squared_risk(y, q[:, None])
except ValueError:
    pass
else:
    raise AssertionError("a broadcasted pairwise difference is not paired loss")
print("Observed squared risk:", empirical_squared_risk(y, q))


## Problem 3. Choose an action under unequal costs

**Problem.** For probabilities $(0.1,0.3,0.6,0.9)$, compare the optimal binary actions under equal error costs and under false-negative cost four with false-positive cost one.

**Solution.** Predicting one incurs conditional cost $c_{\rm FP}(1-q)$, while predicting zero incurs $c_{\rm FN}q$. Choose one when $q\geq c_{\rm FP}/(c_{\rm FP}+c_{\rm FN})$, including ties. Equal costs give threshold $1/2$ and actions $(0,0,1,1)$. Costs one and four give threshold $1/5$ and actions $(0,1,1,1)$. The probabilities stay fixed; the changed consequences alter the action at probability $0.3$.

In [ ]:
def minimum_cost_action(q, false_positive_cost, false_negative_cost):
    q = np.asarray(q)
    if np.any((q < 0) | (q > 1)) or min(false_positive_cost, false_negative_cost) <= 0:
        raise ValueError("probabilities and positive costs required")
    return (false_positive_cost * (1-q) <= false_negative_cost * q).astype(int)


In [ ]:
p = np.array([0.1, 0.3, 0.6, 0.9])
np.testing.assert_array_equal(minimum_cost_action(p, 1, 1), [0, 0, 1, 1])
np.testing.assert_array_equal(minimum_cost_action(p, 1, 4), [0, 1, 1, 1])
# Controlled change: probabilities fixed, only consequences change.
for cost in [1, 2, 4, 9]:
    print(cost, minimum_cost_action(p, 1, cost))
# Plausible failure: always thresholding at 1/2 ignores changed costs.
wrong = (p >= 0.5).astype(int)
assert not np.array_equal(wrong, minimum_cost_action(p, 1, 4))


## Problem 4. Identify the objects and solve the general cost problem

**Problem.** Identify the task, model, parameters, prediction, and action in the preceding examples. Which quantities were estimated? Derive the optimal action when all four entries of a binary cost table can be nonzero, including cases with a reversed threshold or a constant action.

**Solution.** The prediction task assigns a probability to a binary outcome. The model is an affine score followed by a sigmoid, its parameters are $(w,b)$, and its predictions are the entries of $q$. The action is the binary decision returned by the cost rule. All parameters, targets, and costs were supplied; no fitting or statistical parameter estimation occurred. The empirical risk is an observed summary, and its relation to future risk would require a sampling model and an evaluation protocol.

Let $c_{ay}$ be the cost of action $a$ when the outcome is $y$. Given $q=P(Y=1\mid x)$, the two expected costs are

$$r_0(q)=(1-q)c_{00}+qc_{01},\qquad r_1(q)=(1-q)c_{10}+qc_{11}.$$

Set $A=c_{10}-c_{00}$ and $B=c_{11}-c_{10}-c_{01}+c_{00}$. Choosing action one, including ties, is optimal exactly when $A+Bq\leq0$. Therefore:

- If $B<0$, choose one for $q\geq-A/B$.
- If $B>0$, choose one for $q\leq-A/B$.
- If $B=0$, the decision is independent of $q$: choose one when $A\leq0$ and zero otherwise. If $A=0$, both actions tie for every $q$.

Thresholds outside $[0,1]$ can likewise give a constant action. For $C=\begin{pmatrix}0&2\\1&0\end{pmatrix}$, the usual increasing rule has threshold $1/3$. For $C=\begin{pmatrix}1&0\\0&2\end{pmatrix}$, the direction reverses and action one is optimal for $q\leq1/3$. For $C=\begin{pmatrix}2&2\\1&1\end{pmatrix}$, action one always costs less. The cost table, not the probability alone, determines the direction and even whether a nonconstant threshold is needed.


In [ ]:
def binary_cost_action(q, costs):
    q = np.asarray(q, dtype=float)
    costs = np.asarray(costs, dtype=float)
    if costs.shape != (2, 2) or not np.all(np.isfinite(costs)) or np.any(costs < 0):
        raise ValueError("expected a finite nonnegative 2-by-2 cost table")
    if not np.all(np.isfinite(q)) or np.any((q < 0) | (q > 1)):
        raise ValueError("expected probabilities in [0,1]")
    risk_zero = (1-q)*costs[0, 0] + q*costs[0, 1]
    risk_one = (1-q)*costs[1, 0] + q*costs[1, 1]
    return (risk_one <= risk_zero).astype(int)

p = np.array([0.1, 0.3, 0.6, 0.9])
np.testing.assert_array_equal(binary_cost_action(p, [[0, 2], [1, 0]]), [0, 0, 1, 1])
np.testing.assert_array_equal(binary_cost_action(p, [[1, 0], [0, 2]]), [1, 1, 0, 0])
np.testing.assert_array_equal(binary_cost_action(p, [[2, 2], [1, 1]]), [1, 1, 1, 1])
np.testing.assert_array_equal(binary_cost_action(p, [[0, 4], [1, 0]]), minimum_cost_action(p, 1, 4))
